# TinyHybridNet on Tiny-ImageNet — FP32 & QAT (INT8)

**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)
**Goal:** Train `TinyHybridNet` (Fire-inspired mobile-residual CNN) in FP32,
then apply Quantization-Aware Training (QAT) to obtain an INT8 version.
Report **Top-1** and **Top-5** accuracy for both.

## Notebook layout

1. Configuration & reproducibility
2. Dataset & loaders
3. Model definition (`FireMobileResidual`, `TinyHybridNet`) + auto fuse-map
4. QAT helpers (fuse + prepare)
5. FP32 training
6. QAT training and INT8 conversion
7. Evaluation — Top-1 and Top-5 — for FP32 and INT8
8. Final comparison table & ranking
9. Persist experiment summary
10. (Optional) TorchScript export of the INT8 model

## 1. Configuration & reproducibility

In [ ]:
# Standard library
import os
import copy
from pathlib import Path

# Third-party
import torch
import torch.nn as nn
import torch.optim as optim
import torch.ao.quantization as tq
from torch.utils.data import DataLoader

# Local
from ml_utils import (
    get_system_info,
    evaluate_topk,
    train_and_evaluate,
    register_model,
    MODEL_REGISTRY,
    build_comparison_table,
    create_imagenet_loaders,
    load_model,
    count_parameters,
    create_results_summary,
    setup_experiment,
    build_default_config, 
    get_system_info,
    convert_to_int8,
    build_qat,
    load_best_model,
    disk_mb
)


# Quantization backend — must be set before any QAT prep / convert
torch.backends.quantized.engine = "fbgemm"

SEED = 42
device = setup_experiment(SEED, cudnn_benchmark=True)

config = build_default_config(
    seed=SEED,
    device=device,
    save_dir="./models/tinyhybridnet_qat"
)

NUM_CLASSES = config["num_classes"]

print(get_system_info())

/home/rafael/Documents/dnn_study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cpu', 'torch_version': '2.5.1+cu121', 'cuda_available': False}


## 2. Dataset & loaders

We use Tiny-ImageNet-200 from Kaggle. The train folder is split 90/10 into
train/val with a **seeded generator** so the split is identical on every run.

* Training transforms: light augmentation (random crop + hflip + color jitter)
* Validation transforms: deterministic resize + center crop

In [2]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(
    dataset_path=train_path,
    img_size=config["img_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    train_split=config["train_val_split"],
    seed=config["seed"],
    pin_memory=config["pin_memory"],
    persistent_workers=True,
)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {NUM_CLASSES}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

100%|██████████| 474M/474M [00:17<00:00, 29.0MB/s] 

Extracting files...


Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train
Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model architecture — TinyHybridNet

* **Stem**: 3 → 32 channels (3×3 conv + BN + ReLU)
* **FireMobileResidual blocks**: 1×1 squeeze → 3×3 depthwise → 1×1 expand,
  with a residual add (`FloatFunctional`, quantization-friendly) and a final
  ReLU applied after the add.
* **Head**: `AdaptiveAvgPool2d(1)` → `Linear(256, num_classes)`

### Fuse map — built automatically, not hand-written

Unlike the AlexNet notebook (a flat `Sequential` where `[conv_idx, relu_idx]`
pairs can just be read off), `TinyHybridNet` nests `Conv-BN-ReLU` groups
several levels deep — inside `FireMobileResidual.block` and `.shortcut`.
Hand-writing those module paths is error-prone, so `find_fuse_groups` walks
the **entire** module tree recursively (not just top-level `Sequential`
containers) and detects `Conv2d → BatchNorm2d (→ ReLU)` runs wherever they
occur, fusing `[conv, bn, relu]` where a ReLU follows, and `[conv, bn]`
otherwise. This picks up: the stem (1 group), each block's two
depthwise-separable conv-bn-relu pairs plus its non-ReLU expansion conv-bn
(3 groups/block × 6 blocks), and the conv-bn projection inside any
`shortcut` (4 of the 6 blocks change channels/stride, so 4 more groups) —
22 groups total. The residual-add `ReLU` (applied *after* `skip_add`,
outside `block`) is correctly left unfused, since fusing across the
residual connection isn't valid.

In [3]:
class FireMobileResidual(nn.Module):
    """
    Fire-inspired mobile residual block.

    Combines:
    - Squeeze: 1x1 conv reduces channels
    - Mobile: Depthwise separable conv processes spatially
    - Residual: Skip connection with learned projection

    Args:
        in_ch: Input channels
        out_ch: Output channels
        stride: Stride for depthwise conv (enables downsampling)
        squeeze_ratio: Fraction of output channels used in the squeeze layer
    """

    def __init__(self, in_ch, out_ch, stride=1, squeeze_ratio=0.25):
        super().__init__()

        squeeze_ch = max(16, int(out_ch * squeeze_ratio))

        self.block = nn.Sequential(
            # 1x1 bottleneck
            nn.Conv2d(in_ch, squeeze_ch, 1, bias=False),
            nn.BatchNorm2d(squeeze_ch),
            nn.ReLU(inplace=False),

            # 3x3 depthwise convolution
            nn.Conv2d(
                squeeze_ch, squeeze_ch,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=squeeze_ch,
                bias=False,
            ),
            nn.BatchNorm2d(squeeze_ch),
            nn.ReLU(inplace=False),

            # 1x1 expansion (no ReLU here — applied after the residual add)
            nn.Conv2d(squeeze_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

        # FloatFunctional supports fake-quant insertion on the residual add
        # during QAT/convert. Location moved across torch versions, so
        # resolve it defensively rather than hard-coding one module path.
        try:
            self.skip_add = torch.nn.quantized.FloatFunctional()
        except AttributeError:
            self.skip_add = tq.FloatFunctional()
        self.relu = nn.ReLU(inplace=False)

    def forward(self, x):
        out = self.block(x)
        identity = self.shortcut(x)
        out = self.skip_add.add(out, identity)
        return self.relu(out)


class TinyHybridNet(nn.Module):
    """Efficient hybrid CNN for small images (64x64), QAT-ready."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()

        self.quant = tq.QuantStub()
        self.dequant = tq.DeQuantStub()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=False),
        )

        self.features = nn.Sequential(
            FireMobileResidual(32, 64),              # 64x64
            FireMobileResidual(64, 64),               # 64x64
            FireMobileResidual(64, 128, stride=2),    # 32x32
            FireMobileResidual(128, 128),             # 32x32
            FireMobileResidual(128, 256, stride=2),   # 16x16
            FireMobileResidual(256, 256),              # 16x16
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.quant(x)
        x = self.stem(x)
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.dequant(x)
        return x


def build_tinyhybridnet(num_classes: int = NUM_CLASSES) -> nn.Module:
    return TinyHybridNet(num_classes)


def find_fuse_groups(module: nn.Module, prefix: str = "") -> list:
    """Walk the module tree and collect fusable Conv-BN(-ReLU) groups.

    Scans `named_children()` at every level (not just top-level
    `nn.Sequential` containers) so Conv-BN-ReLU runs nested inside
    submodules — like `FireMobileResidual.block` — are also found.
    Returns a list of dotted-path triples/pairs suitable for
    `tq.fuse_modules_qat`, e.g. ["stem.0", "stem.1", "stem.2"].
    """
    groups = []
    children = list(module.named_children())

    i = 0
    while i < len(children):
        name, child = children[i]
        path = f"{prefix}{name}"

        if isinstance(child, nn.Conv2d) and i + 1 < len(children) and isinstance(children[i + 1][1], nn.BatchNorm2d):
            bname, _ = children[i + 1]
            bpath = f"{prefix}{bname}"

            if i + 2 < len(children) and isinstance(children[i + 2][1], nn.ReLU):
                rname, _ = children[i + 2]
                groups.append([path, bpath, f"{prefix}{rname}"])
                i += 3
                continue

            groups.append([path, bpath])
            i += 2
            continue

        # Not a fusable conv start at this level — recurse into the child
        # if it has submodules of its own (covers nested blocks like
        # FireMobileResidual.block / .shortcut).
        if len(list(child.children())) > 0:
            groups.extend(find_fuse_groups(child, prefix=f"{path}."))

        i += 1

    return groups


register_model(
    "tinyhybridnet",
    build_tinyhybridnet,
    fuse_map=find_fuse_groups(build_tinyhybridnet()),
    lr=config["lr_from_scratch"],
)

print(f"Fuse map ({len(MODEL_REGISTRY['tinyhybridnet']['fuse_map'])} groups):")
for g in MODEL_REGISTRY["tinyhybridnet"]["fuse_map"]:
    print(" ", g)


# Smoke test — verify forward pass shape
def _sanity_check():
    x = torch.randn(2, 3, config["img_size"], config["img_size"])
    m = build_tinyhybridnet()
    m.eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, NUM_CLASSES), y.shape
    print(f"  TinyHybridNet OK -> {tuple(y.shape)}, params={count_parameters(m)/1e6:.2f}M")


_sanity_check()

Fuse map (22 groups):
  ['stem.0', 'stem.1', 'stem.2']
  ['features.0.block.0', 'features.0.block.1', 'features.0.block.2']
  ['features.0.block.3', 'features.0.block.4', 'features.0.block.5']
  ['features.0.block.6', 'features.0.block.7']
  ['features.0.shortcut.0', 'features.0.shortcut.1']
  ['features.1.block.0', 'features.1.block.1', 'features.1.block.2']
  ['features.1.block.3', 'features.1.block.4', 'features.1.block.5']
  ['features.1.block.6', 'features.1.block.7']
  ['features.2.block.0', 'features.2.block.1', 'features.2.block.2']
  ['features.2.block.3', 'features.2.block.4', 'features.2.block.5']
  ['features.2.block.6', 'features.2.block.7']
  ['features.2.shortcut.0', 'features.2.shortcut.1']
  ['features.3.block.0', 'features.3.block.1', 'features.3.block.2']
  ['features.3.block.3', 'features.3.block.4', 'features.3.block.5']
  ['features.3.block.6', 'features.3.block.7']
  ['features.4.block.0', 'features.4.block.1', 'features.4.block.2']
  ['features.4.block.3', 'feat

## 4. QAT helpers

Reuses `prepare_qat_model` from `ml_utils` (the same single-source-of-truth
helper used in the AlexNet notebook): puts the model in `train()` mode, sets
the fbgemm `qconfig`, fuses the groups from `find_fuse_groups`, and returns
`prepare_qat(model)`.

## 5. FP32 training

In [5]:
fp32_models = {}
fp32_training_results = {}

criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

for name, spec in MODEL_REGISTRY.items():
    model = spec["ctor"]().to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=spec["lr"],
        weight_decay=config["weight_decay"],
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config["epochs_fp32"],
    )

    results = train_and_evaluate(
        model=model,
        model_name=name,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        epochs=config["epochs_fp32"],
        scheduler=scheduler,
        device=device,
        save_dir=config["save_dir"],
        use_amp=config["use_amp"],
        early_stopping_patience=config["early_stopping_patience"],
        model_ctor=spec["ctor"],
    )

    fp32_models[name] = model
    fp32_training_results[name] = results

Training: tinyhybridnet (lr=0.0003, epochs=2)


/home/rafael/Documents/dnn_study/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
Training:   0%|          | 0/1406 [00:00<?, ?it/s]/home/rafael/Documents/dnn_study/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Training:   4%|▎         | 51/1406 [01:12<32:05,  1.42s/it, loss=5.3107, acc=0.49%]



[!] Training interrupted by user. Best checkpoint so far is on disk as tinyhybridnet_best.pth.


ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x787188c48e30>> (for post_run_cell), with arguments args (<ExecutionResult object at 7871926afb90, execution_count=5 error_before_exec=None error_in_exec=Connection lost info=<ExecutionInfo object at 7871926afad0, raw_cell="fp32_models = {}
fp32_training_results = {}

crite.." transformed_cell="fp32_models = {}
fp32_training_results = {}

crite.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/rafael/Documents/dnn_study/tinyhybridnet_qat.ipynb#X13sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## 6. Quantization-Aware Training (QAT)

QAT fine-tuning happens on GPU; `convert()` and INT8 inference must run on
CPU. We use an `epoch_callback` (passed straight to `train_model`) to freeze
BatchNorm running stats and disable the fake-quant observers in the later
epochs, per `config["qat_freeze_bn_epoch"]` / `config["qat_disable_observer_epoch"]`.

> Note: results are stored under the **plain architecture name**
> (`qat_training_results[name]`), matching how the AlexNet notebook's
> `qat_training_results` dict is keyed — `f"qat_{name}"` is only used for
> the checkpoint filename prefix, not the dict key. Any downstream analysis
> script reading `experiment_summary.json` should look up QAT results by
> the plain name, not `"qat_" + name`.

In [ ]:
def make_freeze_callback(freeze_bn_epoch: int, disable_observer_epoch: int):
    def callback(epoch: int, model: nn.Module) -> None:
        if epoch == freeze_bn_epoch:
            model.apply(torch.nn.intrinsic.qat.freeze_bn_stats)
        if epoch == disable_observer_epoch:
            model.apply(torch.ao.quantization.disable_observer)
    return callback


qat_models = {}
qat_training_results = {}

criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

for name in MODEL_REGISTRY:
    qat_model = build_qat(name)

    optimizer = optim.AdamW(
        qat_model.parameters(),
        lr=config["lr_qat"],
        weight_decay=config["weight_decay"],
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config["epochs_qat"],
    )

    results = train_and_evaluate(
        qat_model,
        train_loader,
        val_loader,
        optimizer,
        criterion,
        num_epochs=config["epochs_qat"],
        device=device,
        scheduler=scheduler,
        save_dir=config["save_dir"],
        model_name=f"qat_{name}",
        use_amp=False,
        early_stopping_patience=config["early_stopping_patience"],
        epoch_callback=make_freeze_callback(
            config["qat_freeze_bn_epoch"],
            config["qat_disable_observer_epoch"],
        ),
    )

    qat_models[name] = qat_model
    qat_training_results[name] = results

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
Evaluation: 100%|██████████| 157/157 [00:05<00:00, 28.87it/s, acc=9.88%]


Error: You must call wandb.init() before wandb.log()

## 7. INT8 conversion and CPU evaluation

`tq.convert` materialises true quantised ops; this must run on CPU with the
model in `eval()` mode. We build a small CPU-side val loader
(`num_workers=0`) and reuse `evaluate_topk` for Top-1/Top-5.

In [ ]:
val_loader_cpu = DataLoader(
    val_ds,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

cpu_device = torch.device("cpu")

int8_models = {}

for name in MODEL_REGISTRY:
    model = qat_models[name].cpu().eval()
    int8_models[name] = convert_to_int8(model)

for name, m in int8_models.items():
    torch.save(m.state_dict(), os.path.join(config["save_dir"], f"{name}.pth"))

print("INT8 conversion done.")

int8_metrics = {}

for name, m in int8_models.items():
    m = m.cpu().eval()

    metrics = evaluate_topk(m, val_loader_cpu, criterion, cpu_device)
    int8_metrics[name] = metrics

    print(
        f"{name:22s} | loss={metrics['loss']:.4f} | "
        f"top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%"
    )

## 8. FP32 evaluation — reload best checkpoints

Reload the best FP32 checkpoint from disk to make sure we're evaluating the
*best* epoch, not whatever was in memory at the end of training.

In [ ]:
fp32_models = {
    name: load_best_model(name, spec["ctor"])
    for name, spec in MODEL_REGISTRY.items()
}

fp32_metrics = {}
for name, m in fp32_models.items():
    metrics = evaluate_topk(m, val_loader, criterion, device)
    fp32_metrics[name] = metrics
    print(
        f"{name:22s} | loss={metrics['loss']:.4f} | "
        f"top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%"
    )

## 9. Final comparison — Top-1, Top-5, size, params

* **Top-1 / Top-5 accuracy** on the validation split
* **Trainable parameters** (millions)
* **Disk size** of the checkpoint (MB) — for INT8 this is the *real*
  on-disk size

In [ ]:
rows = []

for name, m in fp32_models.items():
    rows.append({
        "model": name,
        "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss": fp32_metrics[name]["loss"],
        "params_M": count_parameters(m) / 1e6,
        "size_MB": disk_mb(os.path.join(config["save_dir"], f"{name}_best.pth")),
    })

for name, m in int8_models.items():
    rows.append({
        "model": f"{name}_INT8",
        "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss": int8_metrics[name]["loss"],
        # Converted INT8 module weights are packed; report the float-equivalent
        # param count from the matching FP32 model for reference.
        "params_M": count_parameters(fp32_models[name]) / 1e6,
        "size_MB": disk_mb(os.path.join(config["save_dir"], f"{name}.pth")),
    })

df = build_comparison_table(rows)
df.to_csv(os.path.join(config["save_dir"], "final_comparison.csv"), index=False)
df

print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(
        f"{i+1:2d}. {row['model']:22s} [{row['precision']}] | "
        f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
        f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB"
    )

## 10. Persist experiment summary

A single JSON next to the checkpoints captures: config, system info, and
per-model metrics. Re-running this notebook with the same seed should
reproduce these numbers up to non-determinism in cuDNN kernels.

In [ ]:
summary_results = {
    "fp32_metrics": fp32_metrics,
    "int8_metrics": int8_metrics,
    "fp32_training_results": fp32_training_results,
    "qat_training_results": qat_training_results,
}

create_results_summary(
    model=fp32_models["tinyhybridnet"],
    results=summary_results,
    config=config.to_dict(),
    system_info=get_system_info(),
    output_path=os.path.join(config["save_dir"], "experiment_summary.json"),
)

print(f"Summary written to: {os.path.join(config['save_dir'], 'experiment_summary.json')}")

## 11. (Optional) TorchScript export of the INT8 model

Quantized models with grouped/depthwise convs can be brittle under
`torch.jit.script`, so tracing with a representative input is the more
reliable export path here.

In [ ]:
example_input = torch.randn(1, 3, config["img_size"], config["img_size"])

for name, m in int8_models.items():
    m = m.cpu().eval()
    traced = torch.jit.trace(m, example_input)
    ts_path = os.path.join(config["save_dir"], f"{name}_int8_ts.pt")
    traced.save(ts_path)
    print(f"Saved TorchScript INT8 model: {ts_path}")